In [3]:
%matplotlib inline
import pandas as pd
import numpy as np
rng = np.random.default_rng()
import matplotlib.pyplot as plt
import matplotlib as mpl
from skimage.measure import block_reduce
from joblib import Parallel, delayed
import time

In [21]:
def multivariate_lognormal_cascade(n, sigma1=.1, sigma2=.1, rho=0):

    mu1 = -1/2 * sigma1**2
    mu2 = -1/2 * sigma2**2
 
    PQ = np.array([1,1], ndmin=3)

    for i in range(n):
        iid = rng.multivariate_normal(np.array([0,0]),  np.array([[1,0], [0, 1]]), (2 * PQ.shape[0], 2 * PQ.shape[0]))
        mul = np.exp(np.stack([mu1 + sigma1*iid[:,:,0], mu2 + sigma2*(rho*iid[:,:,0] + (1-rho**2)**0.5 * iid[:,:,1])],axis=2))
        PQ = np.kron(PQ, np.ones((2,2,1))) * mul

    return(PQ / PQ.sum(axis=(0,1)))


def theil(pop):
  tp = pop.sum(axis=2)
  prob = pop[:,:,:] / tp[:,:, np.newaxis]
  ep = np.sum(-np.log2(prob) * prob, axis=2)
  e = (ep * tp / tp.sum()).sum()
  return(1 - e)

def scale_theil(pop, scales):
    stheils = []
    for s in scales:
      small_arr = block_reduce(pop, block_size=(s,s,1), func=np.sum)
      stheils.append(theil(small_arr))
    return(stheils)

In [27]:
def multivariate_lognormal_cascade(sigma1s=[0.1,0.1,0.1,0.1,0.3,0.3,0.3], sigma2s=[0.1,0.1,0.1,0.1,0.3,0.3,0.3], rhos=[0,0,0,0,0,0,0]):

    
 
    PQ = np.array([1,1], ndmin=3)

    for sigma1, sigma2, rho in zip(sigma1s, sigma2s, rhos):

        mu1 = -1/2 * sigma1**2
        mu2 = -1/2 * sigma2**2
        iid = rng.multivariate_normal(np.array([0,0]),  np.array([[1,0], [0, 1]]), (2 * PQ.shape[0], 2 * PQ.shape[0]))
        mul = np.exp(np.stack([mu1 + sigma1*iid[:,:,0], mu2 + sigma2*(rho*iid[:,:,0] + (1-rho**2)**0.5 * iid[:,:,1])],axis=2))
        PQ = np.kron(PQ, np.ones((2,2,1))) * mul

    return(PQ / PQ.sum(axis=(0,1)))

In [28]:
toto = multivariate_lognormal_cascade()

In [89]:
toto = multivariate_lognormal_cascade()
scale_theil(toto, [1,2,4,8,16,32,64,128])

[0.10435872735909235,
 0.08257874874741467,
 0.05537597744604872,
 0.024411629099271748,
 0.011987623564398753,
 0.006475931865181694,
 0.0027970971352566654,
 0.0]

In [108]:
sigma1s=[0.3,0.3,0.3,0.1,0.1,0.1,0.1]
sigma2s=[0.3,0.3,0.3,0.1,0.1,0.1,0.1]
toto =multivariate_lognormal_cascade(sigma1s=sigma1s, sigma2s=sigma2s, rhos=[0,0,0,0,0,0,0])
scale_theil(toto, [1,2,4,8,16,32,64,128])

[0.1033800577669457,
 0.1010392174130077,
 0.09808830438817107,
 0.09500259460230209,
 0.09121384109951203,
 0.06954949925542997,
 0.010657878777346141,
 0.0]